In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from translast.modeling.albert import AlbertConfig, AlbertModel, AlbertMaskedWrapper, AlbertMaskedTrainer
from translast.modeling.train import TrainerConfig, Trainer
from translast.config import MODELS_DIR, PROCESSED_DATA_DIR

2025-03-08 23:04:52.153 | INFO     | translast.config:<module>:11 - PROJ_ROOT path is: /Users/achamorro/Library/CloudStorage/OneDrive-TexasA&MUniversity/Development/Python/transLast


In [3]:
import typer
import json
from loguru import logger
from tqdm import tqdm

import torch

In [29]:
logger.info("Training ALBERT model...")
trainer_config = TrainerConfig.from_json("../translast/modeling/trainer_config.json")
albert_config = AlbertConfig.from_json("../translast/modeling/albert_config.json")
model = AlbertModel(albert_config)
data_iter = ...  # Replace with your data iterator
# save_dir = "models"
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# trainer = Trainer(trainer_config, model, data_iter, save_dir, device)
# trainer.train(loss_function, data_parallel=True)
# for i in tqdm(range(10), total=10):
#     if i == 5:
#         logger.info("Something happened for iteration 5.")
logger.success("Modeling training complete.")
# # -----------------------------------------

2025-03-09 00:01:46.181 | INFO     | __main__:<module>:1 - Training ALBERT model...
2025-03-09 00:01:46.191 | SUCCESS  | __main__:<module>:13 - Modeling training complete.


In [5]:
def create_padding_mask(seq, pad_token=0):
    mask = (seq == pad_token).unsqueeze(1).unsqueeze(2)
    return mask  # (batch_size, 1, 1, seq_len)

input_ids = torch.randint(0, albert_config.vocab_size, (trainer_config.batch_size, albert_config.max_position_embeddings))
mask = create_padding_mask(torch.tensor([[1, 2, 0, 0], [3, 4, 5, 0]]))

input_ids

tensor([[19380,  7944,  5762,  ..., 25803, 20588, 29920],
        [27861, 28178, 28419,  ..., 12038, 20772, 29801],
        [ 1204,  1595, 20591,  ..., 17411, 15616, 29046],
        ...,
        [26431, 15636,  3767,  ..., 22184,  8262,  3216],
        [15441, 30041,  4127,  ..., 11439,  7574, 10493],
        [ 3471, 28319, 27249,  ...,  3808, 18598, 21455]])

In [6]:
def create_attention_mask(batch_size, seq_length, mask_percentage):
    # Create a mask with the given percentage of masked positions
    mask = torch.rand((batch_size, seq_length)) < mask_percentage
    key_padding_mask = mask.float()
    
    # Create masked positions based on the mask
    masked_pos = mask.nonzero(as_tuple=False)
    
    return key_padding_mask, masked_pos
mask_percentage = 0.15  # 15% of positions will be masked

In [18]:
# Example usage
wrapper = AlbertMaskedWrapper(model)

# Example input data
input_ids = torch.randint(0, albert_config.vocab_size, (trainer_config.batch_size, albert_config.max_position_embeddings))
token_type_ids = torch.zeros((trainer_config.batch_size, albert_config.max_position_embeddings), dtype=torch.long)
key_padding_mask, masked_pos = create_attention_mask(trainer_config.batch_size, albert_config.max_position_embeddings, mask_percentage)

# Forward pass
cls_logits, token_logits = wrapper(input_ids, token_type_ids, key_padding_mask.bool(), masked_pos)
print(cls_logits.shape)  # Should be (batch_size, 2)
print(token_logits.shape)  # Should be (batch_size, num_masked_positions, vocab_size)

torch.Size([128, 2])
torch.Size([128, 512, 30522])


In [19]:
import torch
import torch.nn.functional

import datasets
import tokenizers
import transformers
import transformers.data.data_collator

import numpy as np

import tqdm
import argparse
import datetime
import functools
from pathlib import Path

/Users/achamorro/miniconda3/envs/translast/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [20]:
log = lambda s: print(f"> {s}") # TODO: Colour!

"""
    Create workspace and return pointers 
    to the created directories
"""
def create_workspace(save_id):
    runs_dir = Path("runs")
    root_dir = runs_dir / f"pretrain-{save_id}"
    chk_dir = root_dir / "checkpoints"
    log_dir = root_dir / "log_dir"

    runs_dir.mkdir(exist_ok=True)
    root_dir.mkdir(exist_ok=True)
    chk_dir.mkdir(exist_ok=True)
    log_dir.mkdir(exist_ok=True)
    
    return root_dir, chk_dir, log_dir

"""
    Loading English Wikipedia dataset
    1. Download dataset
    2. Remove title column
    3. Chunk text into smaller sequences
    4. Filter shorter sequences
    5. Create train-test split
"""
def load_dataset(tokenizer, small=False, chunk_size=2000, test_split=.1, builder = '20220301.en', nb_workers = 4):
    log("Loading English Wikipedia dataset.")
    wiki = datasets.load_dataset('wikipedia', builder, split='train[:1%]' if small else 'train')
    wiki.remove_columns("title")
    dataset = wiki

    def _chunk_text(batch, chunk_size=2000):
        chunks = []
        for s in batch['text']:
            chunks += [s[i:i+chunk_size] for i in range(0, len(s), chunk_size)]
        return {'chunks': chunks}
    log("Chunking text to maximum sequence_length")
    dataset = dataset.map(functools.partial(_chunk_text, chunk_size=chunk_size), batched=True, num_proc=nb_workers, remove_columns=dataset.column_names)
    log("Filtering short sequences")
    dataset = dataset.filter(lambda e: len(e['chunks']) >= chunk_size, num_proc=nb_workers)

    log("Creating train-eval split")
    dataset = dataset.train_test_split(test_size=test_split)

    return dataset

"""
    Load pretrained tokenizer
"""
def load_tokenizer():
    tokenizer = transformers.AlbertTokenizer.from_pretrained('albert-base-v2')
    return tokenizer

"""
    Given a batch, process it into a batch of tensors
    1. Pick a random split
    2. Pick a random ordering
    3. Tokenize sentence pairs
    4. Mask tokens
    5. Cast to tensors

    TODO: Generally, make this more efficient
"""
def process_batch(batch, tokenizer, collator, mask_prob, chunk_size=2000, seq_len=1024):
    chunks = batch['chunks']
    batch_size = len(chunks)
    random_deltas = torch.randint(-chunk_size // 4, chunk_size // 4, (batch_size,))

    mid = chunk_size // 2

    sentence_pairs = [(c[:mid+random_deltas[i]], c[mid+random_deltas[i]:]) for i, c in enumerate(chunks)]
    ordering = torch.randint(0, 2, (batch_size,))

    sentence_pairs = [(c[0],c[1]) if ordering[i] else (c[1], c[0]) for i, c in enumerate(sentence_pairs)]

    tokenized_pairs = tokenizer([s0 for (s0,_) in sentence_pairs], [s1 for (_,s1) in sentence_pairs], 
                        padding='max_length', max_length=seq_len, pad_to_multiple_of=8, truncation=True,
                        return_special_tokens_mask=True, return_tensors='pt')
    masked_input, token_labels = collator.torch_mask_tokens(tokenized_pairs['input_ids'], tokenized_pairs['special_tokens_mask'])

    return masked_input,\
           tokenized_pairs['token_type_ids'],\
           tokenized_pairs['attention_mask'].bool(),\
           token_labels,\
           ordering

class EpochMetric:
    def __init__(self):
        self.loss = 0.0

        self.cls_loss = 0.0
        self.cls_accuracy = 0.0

        self.token_loss = 0.0
        self.token_accuracy = 0.0

        self.nb_updates = 0

    def update(self, loss, cls, token):
        self.loss += loss

        self.cls_loss += cls[0]
        self.cls_accuracy += cls[1]

        self.token_loss += token[0]
        self.token_accuracy += token[1]

        self.nb_updates += 1

    def __str__(self):
        s = ""
        s += f"loss: {self.loss / self.nb_updates} "
        s += f"| cls: [loss: {self.cls_loss / self.nb_updates}, accuracy: {100.0 * self.cls_accuracy / self.nb_updates:.2f}%] "
        s += f"| token: [loss: {self.token_loss / self.nb_updates}, accuracy: {100.0 * self.token_accuracy / self.nb_updates:.2f}%]"
        return s

In [21]:
device = torch.device(device = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')

log(f"device: {device.type}")

save_id = str(datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S"))

log("Loading pretrained tokenizer")
tokenizer = load_tokenizer()
albert_config.vocab_size = len(tokenizer) + (8 - len(tokenizer) % 8) # pad to multiple of 8 for tensor core optimisation
trainer_config.mask_prob = 0.15
collator = transformers.DataCollatorForLanguageModeling(tokenizer, mlm=True, mlm_probability=trainer_config.mask_prob)

log("Loading Wikipedia Dataset")
trainer_config.chunk_size = 2000
trainer_config.test_size = 0.1
trainer_config.small = True
# TODO: Convert to dataloader version
dataset = load_dataset(tokenizer, small=trainer_config.small, chunk_size=trainer_config.chunk_size, test_split=trainer_config.test_size)


> device: mps
> Loading pretrained tokenizer
> Loading Wikipedia Dataset
> Loading English Wikipedia dataset.
> Chunking text to maximum sequence_length
> Filtering short sequences
> Creating train-eval split


In [22]:
idx = np.random.choice(len(dataset['train']), trainer_config.batch_size)
b = dataset['train'][idx[1*trainer_config.mini_batch_size:(2)*trainer_config.mini_batch_size]]

t = process_batch(b, tokenizer, collator, trainer_config.mask_prob, seq_len=albert_config.max_position_embeddings)

Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.


In [23]:
trainer = AlbertMaskedTrainer(albert_config, trainer_config)
metrics = trainer.train_step(t)

In [24]:
def train_epoch(trainer, dataset, tokenizer, collator, config, args):
    epoch_metrics = EpochMetric()
    update_frequency = config.batch_size // config.mini_batch_size
    pb = tqdm.tqdm(range(32), disable=~args.tqdm)
    
    for _ in pb:
        idx = np.random.choice(len(dataset['train']), config.batch_size)
        
        for i in range(update_frequency):
            b = dataset['train'][idx[i*config.mini_batch_size:(i+1)*config.mini_batch_size]]
            b = process_batch(b, tokenizer, collator, config.mask_prob, seq_len=trainer.config.max_position_embeddings)
            metrics = trainer.train_step(b)
            epoch_metrics.update(*metrics)
        
        if not args.no_save and (trainer.ts + 1) % config.save_frequency == 0:
            trainer.save_checkpoint(args.chk_dir / f"albert-{args.model}-checkpoint-{str(trainer.ts).zfill(7)}.pt")
        
        trainer.ts += 1
        display = f"training | ts: {str(trainer.ts).zfill(7)} | {str(epoch_metrics)}"
        pb.set_description(display)
        pb.update(1)
    
    if not args.tqdm:
        log(display)
    
    return epoch_metrics

def evaluate_epoch(trainer, dataset, tokenizer, collator, config, args):
    epoch_metrics = EpochMetric()
    pb = tqdm.tqdm(range(4 * (config.batch_size // config.mini_batch_size)), disable=~args.tqdm)
    
    for _ in pb:
        idx = np.random.choice(len(dataset['test']), config.mini_batch_size)
        b = dataset['test'][idx]
        b = process_batch(b, tokenizer, collator, config.mask_prob, seq_len=trainer.config.max_position_embeddings)
        metrics = trainer.eval_step(b)
        epoch_metrics.update(*metrics)
        
        display = f"evaluation | ts: {str(trainer.ts).zfill(7)} | {str(epoch_metrics)}"
        pb.set_description(display)
        pb.update(1)
    
    if not args.tqdm:
        log(display)
    
    return epoch_metrics

In [31]:
import os

os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

from argparse import Namespace

args = Namespace(tqdm=True, small=True, model='base', no_save=False)

save_id = str(datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S"))
if not args.no_save:
    args.root_dir, args.chk_dir, args.log_dir = create_workspace(save_id)


log("Loading pretrained tokenizer")
tokenizer = load_tokenizer()
albert_config.vocab_size = len(tokenizer) + (8 - len(tokenizer) % 8) # pad to multiple of 8 for tensor core optimisation
trainer_config.mask_prob = 0.15
collator = transformers.DataCollatorForLanguageModeling(tokenizer, mlm=True, mlm_probability=trainer_config.mask_prob)

log("Loading Wikipedia Dataset")
trainer_config.chunk_size = 2000
trainer_config.test_size = 0.1
trainer_config.small = True
# TODO: Convert to dataloader version
dataset = load_dataset(tokenizer, small=trainer_config.small, chunk_size=trainer_config.chunk_size, test_split=trainer_config.test_size)

trainer = AlbertMaskedTrainer(albert_config, trainer_config)
trainer.ts = 0

> Loading pretrained tokenizer
> Loading Wikipedia Dataset
> Loading English Wikipedia dataset.
> Chunking text to maximum sequence_length
> Filtering short sequences
> Creating train-eval split


In [32]:
while trainer.ts < 125000:
    train_epoch(trainer, dataset, tokenizer, collator, trainer_config, args)
    evaluate_epoch(trainer, dataset, tokenizer, collator, trainer_config, args)

/var/folders/pw/kms117q92j3_c4g17tz9blmh0000gr/T/ipykernel_7337/3071182919.py:4: DeprecationWarning: Bitwise inversion '~' on bool is deprecated. This returns the bitwise inversion of the underlying int object and is usually not what you expect from negating a bool. Use the 'not' operator for boolean negation or ~int(x) if you really want the bitwise inversion of the underlying int.
  pb = tqdm.tqdm(range(32), disable=~args.tqdm)
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longe

KeyboardInterrupt: 

In [35]:
import unicodedata